In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import os

In [ ]:
#salva os dados em um arquivo csv - esse sera o padrao que vamos manter

import json
import pandas as pd
caminho_arquivo = "../results/evaluation_rmse_mae_2PART4.json"
arquivo_saida = "../results/evaluation_rmse_mae_2PART4.csv"

with open(caminho_arquivo, 'r') as arquivo:
    data = json.load(arquivo)
csv_data = []

for key, metrics in data.items():
    parts = key.split()
    imputacao = parts[0]
    tcp = parts[3]
    link = parts[6]
    modelo = parts[-1].replace(",", "")  
    csv_data.append({
        "imputacao": imputacao,
        "link": link,
        "tcp": tcp,
        "modelo": modelo,
        "RMSE": metrics["RMSE"],
        "MAE": metrics["MAE"], 
        "NRMSE":metrics["NRMSE"]
    })

df = pd.DataFrame(csv_data)

df.to_csv(arquivo_saida, index=False)

In [ ]:
def get_link_list(dir):
    linklist = []
    print(dir)
    for file in os.listdir(dir):
        link = file.split(' ')[4]
        linklist.append(link)
    return linklist

In [ ]:
links = get_link_list("../datasets/choosen-best-svd/")

In [ ]:
arquivo = "../results/bi-lstm/evaluation_rmse_mae.csv"
df = pd.read_csv(arquivo)

df_bbr_filtered = df[(df['tcp'] == 'bbr') & (df['modelo'] ==  'LSTM') & (df['link'].isin(links))]

pivot_df = df_bbr_filtered.pivot_table(index=['link', 'tcp', 'modelo'], columns='imputacao', values='RMSE')

if not pivot_df.empty:
    pivot_df.plot(kind='barh', figsize=(10, 15))
    plt.title('Comparação de RMSE LSTM', fontsize=14)
    plt.xlabel('Link e TCP', fontsize=12)
    plt.ylabel('RMSE', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Imputação', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No data available after filtering. Check the filter criteria.")


Compare the models for choosen links based on forecasting performance

In [ ]:
file_path =  "../results/bi-lstm/evaluation_rmse_mae.csv"

df = pd.read_csv(file_path)

# Define the links to filter (best performing SVD on forecasting)
links_to_filter = ['ap-ba', 'ba-pa', 'ce-ro', 'es-pr', 'go-se', 'ma-rj', 'pb-es', 'ro-se']

filtered_df = df[df['link'].isin(links_to_filter)]

# Pivot the data for comparison (models vs. RMSE per link)
pivot_df = filtered_df.pivot_table(
    index='link',
    columns='modelo',
    values='RMSE'
)

# Group by model and calculate the average RMSE for each model
average_rmse_by_model = filtered_df.groupby('modelo')['RMSE'].mean()

# Determine the best model (lowest average RMSE)
best_model = average_rmse_by_model.idxmin()
best_model_rmse = average_rmse_by_model.min()

print("Average RMSE by Model:")
print(average_rmse_by_model)
print(f"\nBest Model: {best_model} with RMSE: {best_model_rmse:.4f}")

if not pivot_df.empty:
    pivot_df.plot(kind='bar', figsize=(12, 8))
    plt.title('Comparison of RMSE by Model', fontsize=16)
    plt.xlabel('Link', fontsize=12)
    plt.ylabel('RMSE', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No data available after filtering. Check the filter criteria.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from pathlib import Path


def analyze_models_from_csv(csv_path):
    """
    Analyze model performance comparing all imputation methods
    
    Parameters:
    csv_path (str): Path to the CSV file
    """
    # Read data
    try:
        df = pd.read_csv(csv_path)
        print("Data loaded successfully!")
        print("\nDataset Shape:", df.shape)
        print("\nAvailable imputation methods:", df['imputacao'].unique())
        print("\nFirst few rows:")
        print(df.head())
    except Exception as e:
        print(f"Error reading CSV file: {e}")
        return
    
    # Create output directories
    output_dir = Path('analysis_results')
    plots_dir = output_dir / 'plots'
    for dir_path in [output_dir, plots_dir]:
        dir_path.mkdir(parents=True, exist_ok=True)
    
    def create_heatmap(df, metric):
        """Create heatmap comparing imputation methods and models"""
        plt.figure(figsize=(12, 8))
        pivot_table = df.pivot_table(
            values=metric,
            index='imputacao',
            columns='modelo',
            aggfunc='mean'
        )
        
        sns.heatmap(pivot_table, annot=True, fmt='.2f', cmap='YlOrRd_r',
                   cbar_kws={'label': metric})
        plt.title(f'{metric} Comparison - Imputation Methods vs Models')
        plt.tight_layout()
        plt.savefig(plots_dir / f'{metric}_heatmap.png')
        plt.close()

    def create_comparison_plots(df, metric):
        """Create detailed comparison plots"""
        # Boxplot
        plt.figure(figsize=(15, 8))
        sns.boxplot(data=df, x='modelo', y=metric, hue='imputacao')
        plt.title(f'{metric} Distribution by Model and Imputation Method')
        plt.xticks(rotation=45)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.savefig(plots_dir / f'{metric}_boxplot.png')
        plt.close()
        
        # Violin plot
        plt.figure(figsize=(15, 8))
        sns.violinplot(data=df, x='modelo', y=metric, hue='imputacao')
        plt.title(f'{metric} Distribution (Violin Plot)')
        plt.xticks(rotation=45)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.savefig(plots_dir / f'{metric}_violin.png')
        plt.close()

    def perform_statistical_analysis(df):
        """Perform comprehensive statistical analysis"""
        # Summary statistics
        summary_stats = df.groupby(['imputacao', 'modelo']).agg({
            'RMSE': ['mean', 'std', 'min', 'max'],
            'MAE': ['mean', 'std', 'min', 'max'],
            'NRMSE': ['mean', 'std', 'min', 'max']
        }).round(4)
        
        # ANOVA test for each metric
        anova_results = {}
        for metric in ['RMSE', 'MAE', 'NRMSE']:
            # ANOVA for imputation methods
            imp_groups = [group for _, group in df.groupby('imputacao')[metric]]
            f_stat, p_val = stats.f_oneway(*imp_groups)
            anova_results[f'{metric}_imputation'] = {
                'f_statistic': f_stat,
                'p_value': p_val
            }
            
            # ANOVA for models
            model_groups = [group for _, group in df.groupby('modelo')[metric]]
            f_stat, p_val = stats.f_oneway(*model_groups)
            anova_results[f'{metric}_models'] = {
                'f_statistic': f_stat,
                'p_value': p_val
            }
        
        return summary_stats, anova_results

    def create_ranking_analysis(df):
        """Create ranking analysis of methods and models"""
        rankings = pd.DataFrame()
        
        for metric in ['RMSE', 'MAE', 'NRMSE']:
            # Calculate mean performance for each combination
            mean_perf = df.groupby(['imputacao', 'modelo'])[metric].mean().reset_index()
            
            # Rank combinations for each metric (lower is better)
            mean_perf[f'{metric}_rank'] = mean_perf[metric].rank()
            
            if rankings.empty:
                rankings = mean_perf
            else:
                rankings = rankings.merge(mean_perf[['imputacao', 'modelo', f'{metric}_rank']], 
                                       on=['imputacao', 'modelo'])
        
        # Calculate average rank across all metrics
        rank_columns = [col for col in rankings.columns if col.endswith('_rank')]
        rankings['average_rank'] = rankings[rank_columns].mean(axis=1)
        rankings = rankings.sort_values('average_rank')
        
        return rankings

    # Perform analyses
    print("\nPerforming analysis...")
    
    # Create visualizations for each metric
    metrics = ['RMSE', 'MAE', 'NRMSE']
    for metric in metrics:
        create_heatmap(df, metric)
        create_comparison_plots(df, metric)
    
    # Statistical analysis
    summary_stats, anova_results = perform_statistical_analysis(df)
    rankings = create_ranking_analysis(df)
    
    # # Save results
    # with pd.ExcelWriter(output_dir / 'comprehensive_analysis.xlsx') as writer:
    #     # Summary statistics
    #     summary_stats.to_excel(writer, sheet_name='Summary Statistics')
        
    #     # ANOVA results
    #     anova_df = pd.DataFrame.from_dict(anova_results, orient='index')
    #     anova_df.to_excel(writer, sheet_name='ANOVA Results')
        
    #     # Rankings
    #     rankings.to_excel(writer, sheet_name='Method Rankings')
        
    #     # Raw data
    #     df.to_excel(writer, sheet_name='Raw Data')
    
    # Print key findings
    print("\n=== Key Findings ===")
    
    print("\nTop 5 Best Performing Combinations (Average Rank):")
    print(rankings[['imputacao', 'modelo', 'average_rank']].head())
    
    print("\nBest Method for Each Metric:")
    for metric in metrics:
        best_combo = df.groupby(['imputacao', 'modelo'])[metric].mean().idxmin()
        best_value = df.groupby(['imputacao', 'modelo'])[metric].mean().min()
        print(f"\n{metric}:")
        print(f"Best combination: {best_combo}")
        print(f"Value: {best_value:.4f}")
    
    print("\nANOVA Test Results:")
    for test, results in anova_results.items():
        print(f"\n{test}:")
        print(f"F-statistic: {results['f_statistic']:.4f}")
        print(f"p-value: {results['p_value']:.4f}")

if __name__ == "__main__":
    # Example usage
    csv_path = "../results/bi-lstm/evaluation_rmse_mae.csv"  # Replace with your CSV file path
    analyze_models_from_csv(csv_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_performance_graphs(csv_path):
    """
    Create multiple visualization graphs for model performance
    
    Parameters:
    csv_path (str): Path to the CSV file
    """
    # Read the data
    df = pd.read_csv(csv_path)
    
    # Create output directory
    import os
    if not os.path.exists('graphs'):
        os.makedirs('graphs')
    
    # Set a consistent style and color palette
    # plt.style.use('seaborn')
    color_palette = sns.color_palette("husl", 6)
    
    # 1. Grouped Bar Plot for RMSE
    plt.figure(figsize=(15, 7))
    df_rmse = df.groupby(['imputacao', 'modelo'])['RMSE'].mean().unstack()
    df_rmse.plot(kind='bar', ax=plt.gca())
    plt.title('Average RMSE by Imputation Method and Model', fontsize=16)
    plt.xlabel('Imputation Method', fontsize=12)
    plt.ylabel('RMSE', fontsize=12)
    plt.legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('graphs/rmse_grouped_bar.png')
    plt.close()
    
    # 2. Box Plot for MAE
    plt.figure(figsize=(15, 7))
    sns.boxplot(x='modelo', y='MAE', hue='imputacao', data=df, palette=color_palette)
    plt.title('MAE Distribution by Model and Imputation Method', fontsize=16)
    plt.xlabel('Model', fontsize=12)
    plt.ylabel('MAE', fontsize=12)
    plt.legend(title='Imputation Method', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('graphs/mae_boxplot.png')
    plt.close()
    
    # 3. Violin Plot for NRMSE
    plt.figure(figsize=(15, 7))
    sns.violinplot(x='modelo', y='NRMSE', hue='imputacao', data=df, 
                   split=True, inner="quartile", palette=color_palette)
    plt.title('NRMSE Distribution by Model and Imputation Method', fontsize=16)
    plt.xlabel('Model', fontsize=12)
    plt.ylabel('NRMSE', fontsize=12)
    plt.legend(title='Imputation Method', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.savefig('graphs/nrmse_violin.png')
    plt.close()
    
    # 4. Heatmap of Performance Metrics
    plt.figure(figsize=(16, 10))
    # Pivot table for RMSE
    rmse_pivot = df.pivot_table(values='RMSE', index='imputacao', columns='modelo', aggfunc='mean')
    sns.heatmap(rmse_pivot, annot=True, cmap='YlGnBu', fmt='.2f')
    plt.title('Average RMSE Heatmap', fontsize=16)
    plt.tight_layout()
    plt.savefig('graphs/rmse_heatmap.png')
    plt.close()
    
    # 5. Radar Chart for Comparative Performance
    def make_radar_chart():
        # Prepare data
        metrics = ['RMSE', 'MAE', 'NRMSE']
        
        # Normalize the data
        normalized_df = df.copy()
        for metric in metrics:
            normalized_df[metric] = (df[metric] - df[metric].min()) / (df[metric].max() - df[metric].min())
        
        # Get unique combinations
        combinations = normalized_df.groupby(['imputacao', 'modelo'])[metrics].mean()
        
        # Radar Chart
        plt.figure(figsize=(15, 10))
        
        # Number of variables
        categories = metrics
        N = len(categories)
        
        # Create angles for radar chart
        angles = [n / float(N) * 2 * 3.141593 for n in range(N)]
        angles += angles[:1]
        
        # Initialize the plot
        ax = plt.subplot(111, polar=True)
        
        # Plot each combination
        for i, (index, row) in enumerate(combinations.iterrows()):
            values = row.tolist()
            values += values[:1]
            
            # Plot data
            ax.plot(angles, values, linewidth=1, linestyle='solid', label=f"{index[0]} - {index[1]}")
            ax.fill(angles, values, alpha=0.1)
        
        # Fix axis to go in the right order and start at 12 o'clock
        plt.theta_offset = 3.141593 / 2
        plt.theta_direction = -1
        
        # Draw axis lines for each angle
        plt.xticks(angles[:-1], categories)
        
        plt.title("Multivariate Performance Comparison", size=20, y=1.1)
        plt.legend(loc='center left', bbox_to_anchor=(1.1, 0.5))
        plt.tight_layout()
        plt.savefig('graphs/performance_radar.png')
        plt.close()
    
    make_radar_chart()
    
    # Print summary
    print("Graphs have been generated and saved in the 'graphs' directory:")
    print("1. RMSE Grouped Bar Plot")
    print("2. MAE Box Plot")
    print("3. NRMSE Violin Plot")
    print("4. RMSE Heatmap")
    print("5. Performance Radar Chart")

# Usage
if __name__ == "__main__":
    csv_path = "../results/bi-lstm/evaluation_rmse_mae.csv"  # Replace with your CSV file path
    plot_performance_graphs(csv_path)

In [ ]:
# plotando os resultados de dtw - dado imputado versus originais - longgest intervals

dtw_imputed= '../results/DTW_imputation_original.csv'
dtw_prediction = '../results/DTW_LSTM_GRU.csv'

df = pd.read_csv(dtw_imputed)

df_bbr_filtered = df#[(df['tcp'] == 'bbr') & (df['modelo'] ==  'bi-LSTM') & (df['link'].isin(links))]

pivot_df = df_bbr_filtered.pivot_table(index=['link', 'tcp'], columns='imputation', values='dtw_seasonal')

if not pivot_df.empty:
    pivot_df.plot(kind='barh', figsize=(10, 15))
    plt.title('Comparação de RMSE bi-LSTM', fontsize=14)
    plt.xlabel('Link e TCP', fontsize=12)
    plt.ylabel('RMSE', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Imputação', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No data available after filtering. Check the filter criteria.")


In [ ]:
# plotando os resultados de dtw - dado predito versus originais - longgest intervals

dtw_imputed= '../results/DTW_imputation_original.csv'
dtw_prediction = '../results/DTW_LSTM_GRU.csv'

df = pd.read_csv(dtw_prediction)

df_bbr_filtered = df#[(df['tcp'] == 'bbr') & (df['modelo'] ==  'bi-LSTM') & (df['link'].isin(links))]

pivot_df = df_bbr_filtered.pivot_table(index=['link', 'tcp'], columns='imputation', values='dtw_seasonal')

if not pivot_df.empty:
    pivot_df.plot(kind='barh', figsize=(10, 15))
    plt.title('Comparação de RMSE', fontsize=14)
    plt.xlabel('Link e TCP', fontsize=12)
    plt.ylabel('RMSE', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Imputação', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print("No data available after filtering. Check the filter criteria.")


In [ ]:
pivot_df

In [ ]:
#gerando acuracia das predições

#salvar em um arquivo json a acuracia das predicoes
import os
import pandas as pd
import json

def organizar_arquivos_com_acuracia(raiz):
    files_by_technique = {}

    for pasta_raiz, subpastas, arquivos in os.walk(raiz):
        for arquivo in arquivos:
            if arquivo.endswith('.csv'):
                caminho_arquivo = os.path.join(pasta_raiz, arquivo)
                partes = arquivo.split(" ")
                try:
                    imputacao = partes[2]  # Nome do método de imputação
                    model = partes[1]
                    tc = partes[5]
                    link = partes[8]

                    modelo_tcp_key = f'{tc} {model}'

                    # Carregar o arquivo CSV
                    df = pd.read_csv(caminho_arquivo)

                    # Calcular a acurácia do arquivo
                    accuracy = calculate_accuracy(df)

                    # Criar entrada no dicionário files_by_technique
                    if imputacao not in files_by_technique:
                        files_by_technique[imputacao] = []

                    files_by_technique[imputacao].append({
                        "accuracy": accuracy,
                        "MODELO": model,
                        "TCP": tc,
                        "LINK": link
                    })
                
                except IndexError:
                    # Caso de arquivos que não possuem o mesmo formato esperado
                    continue
                except pd.errors.EmptyDataError:
                    # Arquivo vazio
                    print(f"Arquivo: {arquivo} - O arquivo está vazio, subpasta: {pasta_raiz}")
                except Exception as e:
                    # Tratar outras exceções, se necessário
                    print(f"Arquivo: {arquivo}, subpasta: {pasta_raiz} - Erro: {str(e)}")

    return files_by_technique

# Função para calcular a acurácia (substitua pela sua lógica real)
def calculate_accuracy(df):
    def check_accuracy(row):
        prediction = row['Prediction']
        test = row['Test']

        if prediction < 200 and test < 200:
            return 'r'
        elif 200 <= prediction < 500 and 200 <= test < 500:
            return 'o'
        elif 500 <= prediction < 800 and 500 <= test < 800:
            return 'y'
        elif 800 <= prediction < 1000 and 800 <= test < 1000:
            return 'b'
        elif prediction >= 1000 and test >= 1000:
            return 'g'
        else:
            return 'mismatch'
    
    df['Class'] = df.apply(check_accuracy, axis=1)
    df['Acerto'] = df['Class'] != 'mismatch'

    # Calcular a quantidade de vezes que cada letra aparece
    counts = df['Class'].value_counts()

    # Garantir que todas as letras ('r', 'o', 'y', 'b', 'g') estejam presentes
    for letter in ['r', 'o', 'y', 'b', 'g']:
        if letter not in counts.index:
            counts[letter] = 0

    total_acertos = df['Acerto'].sum()
    total_predictions = len(df)
    acuracia_total = (total_acertos / total_predictions) * 100

    return acuracia_total

#aplicando a funcao nos arquivos de predição
raiz = '../graficos/predicoes/round_2/valores-predicao'
data = organizar_arquivos_com_acuracia(raiz)

# Salvando os dados em um arquivo JSON
with open('../graficos/predicoes/round_2/predicoes_acuracia.json', 'w') as f:
    json.dump(data, f, indent=4)

In [ ]:
#cria um dicionario separando por link, tcp e modelo, para plotagem do grafico de barras posteriormente 
import json

# Caminho do arquivo JSON
arquivo_json = "../graficos/predicoes/round_2/predicoes_acuracia.json"

# Carregar o arquivo JSON para um dicionário
with open(arquivo_json, 'r') as arquivo:
    data = json.load(arquivo)

# Dados de accuracy para os links PR-AM e PA-BA
methods = ['interpolacao-linear', 'knn', 'media-movel', 'mediana-movel']
models = ['BBR LSTM', 'BBR GRU', 'CUBIC LSTM', 'CUBIC GRU']

accuracy_values_PRAM = {method: {model: [] for model in models} for method in methods}
accuracy_values_PABA = {method: {model: [] for model in models} for method in methods}
accuracy_values_MGRS = {method: {model: [] for model in models} for method in methods}
# Iterar sobre os dados do JSON e adicionar as acurácias aos dicionários correspondentes
for group, values in data.items():
    for item in values:
        try:
            accuracy = item['accuracy']
            model = item['MODELO']
            tcp = item['TCP'].upper()  # Convertendo para minúsculas
            link = item['LINK']

            #print(f"Acurácia: {accuracy}, Modelo: {model}, TCP: {tcp}, Link: {link}")

            # Ajuste para formatar corretamente as chaves dos modelos
            model_key = f'{tcp} {model}'
            #print(model_key)
            try:

                if link == 'pr-am' and model_key in models:
                    accuracy_values_PRAM[group][model_key].append(accuracy)
                elif link == 'pa-ba' and model_key in models:
                    accuracy_values_PABA[group][model_key].append(accuracy)
                elif link == 'mg-rs' and model_key in models:
                    accuracy_values_MGRS[group][model_key].append(accuracy)
            except:
                pass
        except KeyError as e:
            print(f"Erro ao processar item no JSON: {e}")

#plotar a accuracy para cada link
# Largura das barras
width = 0.15

# Posição das barras
x = np.arange(len(models))
# Tradução dos métodos para inglês
method_labels = {
    'interpolacao-linear': 'Linear Interpolation',
    'knn': 'KNN',
    'media-movel': 'Moving Average',
    'mediana-movel': 'Moving Median'
}

# Função para adicionar valores nas barras
def autolabel(ax, rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height:.2f}', xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha='center', va='bottom')

# Função para criar e salvar o gráfico para PR-AM
def create_and_save_PRAM_graph():
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = []
    colors = ['#4168E1', '#67CB57', '#FF6961', '#9467bd']  # Azul, Verde, Laranja, Roxo
    for i, method in enumerate(methods):
        try:
            means = [np.mean(accuracy_values_PRAM[method][model]) for model in models]
            bars.append(ax.bar(x + i*width - 1.5*width, means, width, label=method_labels[method], color=colors[i]))
        except KeyError:
            pass
    
    ax.set_ylabel('Accuracy (%)', fontsize=17)
    ax.set_xlabel('', fontsize=14)
    ax.set_title('Accuracy for Forecasting Process of PR-AM', fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=17)
    ax.legend()

    plt.ylim(0, 100)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig("../graficos/predicoes/round_2/PR-AM_accuracy.png")
    plt.close()

# Função para criar e salvar o gráfico para MG-RS
def create_and_save_MGRS_graph():
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = []
    colors = ['#4168E1', '#67CB57', '#FF6961', '#9467bd']  # Azul, Verde, Laranja, Roxo
    for i, method in enumerate(methods):
        try:
            means = [np.mean(accuracy_values_MGRS[method][model]) for model in models]
            bars.append(ax.bar(x + i*width - 1.5*width, means, width, label=method_labels[method], color=colors[i]))
        except KeyError:
            pass

    ax.set_ylabel('Accuracy (%)', fontsize=17)
    ax.set_xlabel('', fontsize=14)
    ax.set_title('Accuracy for Forecasting Process of MG-RS', fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=17)
    ax.legend()

    plt.ylim(0, 100)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig("../graficos/predicoes/round_2\MG-RS_accuracy.png")
    plt.close()

# Função para criar e salvar o gráfico para PA-BA
def create_and_save_PABA_graph():
    fig, ax = plt.subplots(figsize=(10, 6))

    bars = []
    colors = ['#4168E1', '#67CB57', '#FF6961', '#9467bd']  # Azul, Verde, Laranja, Roxo
    for i, method in enumerate(methods):
        try:
            means = [np.mean(accuracy_values_PABA[method][model]) for model in models]
            bars.append(ax.bar(x + i*width - 1.5*width, means, width, label=method_labels[method], color=colors[i]))
        except KeyError:
            pass

    ax.set_ylabel('Accuracy (%)', fontsize=17)
    ax.set_xlabel('', fontsize=14)
    ax.set_title('Accuracy for Forecasting Process of PA-BA', fontsize=18)
    ax.set_xticks(x)
    ax.set_xticklabels(models, fontsize=17)
    ax.legend()

    plt.ylim(0, 100)
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    plt.tight_layout()
    plt.savefig("../graficos/predicoes/round_2/PA-BA_accuracy.png")
    plt.close()

# Criar e salvar os gráficos
create_and_save_PRAM_graph()
create_and_save_PABA_graph()
create_and_save_MGRS_graph()